In [ ]:
# !pip install torch

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
df = pd.read_csv("data/final.csv")

FEATURES = [
    'Points', 'FGM', 'FGA', 'FGM3', 'FGA3',
    'FTM',    'FTA', 'OR',  'DR',   'Ast',
    'TO',     'Stl', 'Blk', 'PF',   'Games'
]

X_raw = df[FEATURES].values.astype(np.float32)
y_raw = df['Target'].values.astype(np.float32)

print(f"Samples: {len(X_raw)}  |  Features: {len(FEATURES)}")
print(f"Class balance: {dict(pd.Series(y_raw.astype(int)).value_counts())}")

In [ ]:
def make_loaders(X_tr, y_tr, X_val, y_val, batch_size=32):
    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)

    def to_ds(X, y):
        return TensorDataset(
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )

    tr_loader = DataLoader(to_ds(X_tr_s, y_tr), batch_size=batch_size, shuffle=True)
    val_X_t   = torch.tensor(X_val_s, dtype=torch.float32)
    return tr_loader, val_X_t

In [ ]:
class MLP(nn.Module):
    """
    Fully-connected feedforward network for binary classification.
    Each hidden layer: Linear -> BatchNorm1d -> ReLU -> Dropout
    Output: single linear logit (use BCEWithLogitsLoss during training)
    """
    def __init__(self, in_dim: int, hidden_dims: list, dropout: float = 0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

In [ ]:
def train_one_fold(
    X_tr, y_tr, X_val, y_val,
    hidden_dims  = [128, 64, 32],
    lr           = 1e-3,
    epochs       = 300,
    dropout      = 0.3,
    weight_decay = 1e-4,
    batch_size   = 32,
    verbose      = False
):
    tr_loader, val_X_t = make_loaders(X_tr, y_tr, X_val, y_val, batch_size)
    val_y_t = torch.tensor(y_val, dtype=torch.float32)

    model     = MLP(X_tr.shape[1], hidden_dims, dropout).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=75, gamma=0.5)

    train_losses, val_losses = [], []
    best_auc, best_state     = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        ep_losses = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            ep_losses.append(loss.item())
        scheduler.step()
        train_losses.append(float(np.mean(ep_losses)))

        model.eval()
        with torch.no_grad():
            val_logits = model(val_X_t.to(device)).cpu()
            val_loss   = criterion(val_logits, val_y_t).item()
            val_losses.append(val_loss)
            val_probs  = torch.sigmoid(val_logits).numpy()

        try:
            auc = roc_auc_score(y_val, val_probs)
        except ValueError:
            auc = 0.0
        if auc > best_auc:
            best_auc   = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if verbose and epoch % 50 == 0:
            acc = accuracy_score(y_val, (val_probs > 0.5).astype(float))
            print(f"  Epoch {epoch:3d} | train_loss={train_losses[-1]:.4f}  "
                  f"val_loss={val_losses[-1]:.4f} | val_acc={acc:.4f}  val_auc={auc:.4f}")

    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(val_X_t.to(device))).cpu().numpy()

    final_acc = accuracy_score(y_val, (val_probs > 0.5).astype(float))
    final_auc = roc_auc_score(y_val, val_probs)
    return final_acc, final_auc, train_losses, val_losses

In [ ]:
HIDDEN_DIMS = [128, 64, 32]
LR          = 1e-3
DROPOUT     = 0.3

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_accs, fold_aucs = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_raw, y_raw), 1):
    acc, auc, _, _ = train_one_fold(
        X_raw[tr_idx], y_raw[tr_idx],
        X_raw[val_idx], y_raw[val_idx],
        hidden_dims=HIDDEN_DIMS, lr=LR, dropout=DROPOUT,
        epochs=300, verbose=False
    )
    fold_accs.append(acc)
    fold_aucs.append(auc)
    print(f"Fold {fold}: Acc={acc:.4f}  AUC={auc:.4f}")

print()
print(f"Mean Acc : {np.mean(fold_accs):.4f} +/- {np.std(fold_accs):.4f}")
print(f"Mean AUC : {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}")

In [ ]:
tr_idx, val_idx = next(iter(cv.split(X_raw, y_raw)))

print("Training fold 1 (verbose)...")
_, _, tr_losses, val_losses = train_one_fold(
    X_raw[tr_idx], y_raw[tr_idx],
    X_raw[val_idx], y_raw[val_idx],
    hidden_dims=HIDDEN_DIMS, lr=LR, dropout=DROPOUT,
    epochs=300, verbose=True
)

plt.figure(figsize=(9, 4))
plt.plot(tr_losses,  label="Train BCE Loss", color="steelblue",  linewidth=1.8)
plt.plot(val_losses, label="Val BCE Loss",   color="tomato",     linewidth=1.8, linestyle="--")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("BCE Loss", fontsize=12)
plt.title(f"MLP Training Curve — arch={HIDDEN_DIMS}, dropout={DROPOUT}, lr={LR}", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("training_curve.png", dpi=150)
plt.show()
print("Saved -> training_curve.png")

In [ ]:
summary = {
    "Random Baseline (50/50)"  : (0.5000, 0.5000),
    f"MLP {HIDDEN_DIMS} (ours)": (np.mean(fold_accs), np.mean(fold_aucs)),
}

print(f"{'Model':<35} {'Accuracy':>10} {'AUC-ROC':>10}")
print("-" * 57)
for name, (acc, auc) in summary.items():
    print(f"{name:<35} {acc:>10.4f} {auc:>10.4f}")